# Feature Engineering V3

## Objectif
Préparer un dataset simple pour prédire **`applied_t1`**, le nombre de demandes à T+1.

Architecture :
**historique du segment + poids du corridor + contexte pays/métier + IDMC passé + décisions passées**.



In [70]:
# ============================================================
# 1. IMPORTS ET PARAMÈTRES
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)

INPUT_PATH = Path("../data/consolidated/dataset_demandes_consolide_detail.csv")

# Fichiers produits par ce notebook.
OUTPUT_PATH = Path("../data/feature_engineering_outputs/dataset_ml_features.csv")
QUALITY_PATH = Path("../data/feature_engineering_outputs/data_quality_report.csv")


In [45]:
# ============================================================
# 2. CHARGEMENT DES DONNÉES
# ============================================================

# Charger le fichier consolidé.
df = pd.read_csv(INPUT_PATH)

# Garder une copie des données d'origine en mémoire.
df_source = df.copy()

print("Nombre de lignes :", len(df))
print("Nombre de colonnes :", df.shape[1])
display(df.head())


Nombre de lignes : 120597
Nombre de colonnes : 50


,_row_id,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,procedure_type,app_type,dec_level,app_pc,applied,_demand_row_id,has_asylum_application_data,decisions_recognized,decisions_other,decisions_rejected,decisions_closed,decisions_total,decisions_source_rows,recognition_rate,rejection_rate,has_decisions_data,returned_refugees,resettlement,naturalisation,returned_idps,solutions_source_rows,has_solutions_data,origin_iso,origin_iso2,origin_country,origin_country_fr,origin_region,origin_region_fr,origin_major_area,origin_major_area_fr,asylum_iso,asylum_iso2,asylum_country,asylum_country_fr,asylum_region,asylum_region_fr,asylum_major_area,asylum_major_area_fr,idmc_total_origin,has_idmc_data
0,0,2006,2,Afghanistan,AFG,AFG,11,Australia,AUL,AUS,G,A,AR,C,14,0,1,20.0000,0.0000,0.0000,5.0000,25.0000,1.0000,0.8000,0.0000,1,NaN,"2,048.0000",NaN,NaN,1.0000,1,AFG,AF,Afghanistan,Afghanistan,Southern Asia,Sud de l'Asie centrale,Asia,Asie,AUS,AU,Australia,Australie,Australia-New Zealand,Australie-Nouvelle-Z,Oceania,Oc,NaN,0
1,1,2006,3,Albania,ALB,ALB,11,Australia,AUL,AUS,G,A,AR,C,21,1,1,5.0000,0.0000,13.0000,0.0000,18.0000,1.0000,0.2778,0.7222,1,NaN,NaN,NaN,NaN,NaN,0,ALB,AL,Albania,Albanie,Southern Europe,Europe australe,Europe,Europe,AUS,AU,Australia,Australie,Australia-New Zealand,Australie-Nouvelle-Z,Oceania,Oc,NaN,0
2,2,2006,4,Algeria,ALG,DZA,11,Australia,AUL,AUS,G,A,AR,C,5,2,1,0.0000,0.0000,5.0000,0.0000,5.0000,1.0000,0.0000,1.0000,1,NaN,NaN,NaN,NaN,NaN,0,DZA,DZ,Algeria,Alg,Northern Africa,Afrique septentrionale,Africa,Afrique,AUS,AU,Australia,Australie,Australia-New Zealand,Australie-Nouvelle-Z,Oceania,Oc,NaN,0
3,3,2006,8,Egypt,ARE,EGY,11,Australia,AUL,AUS,G,A,AR,C,38,3,1,28.0000,0.0000,20.0000,0.0000,48.0000,1.0000,0.5833,0.4167,1,NaN,25.0000,NaN,NaN,1.0000,1,EGY,EG,Egypt,Egypte,Northern Africa,Afrique septentrionale,Africa,Afrique,AUS,AU,Australia,Australie,Australia-New Zealand,Australie-Nouvelle-Z,Oceania,Oc,NaN,0
4,4,2006,14,Bahrain,BAH,BHR,11,Australia,AUL,AUS,G,A,AR,C,11,4,1,11.0000,0.0000,0.0000,0.0000,11.0000,1.0000,1.0000,0.0000,1,NaN,NaN,NaN,NaN,NaN,0,BHR,BH,Bahrain,Bahre,Western Asia,Asie occidentale,Asia,Asie,AUS,AU,Australia,Australie,Australia-New Zealand,Australie-Nouvelle-Z,Oceania,Oc,NaN,0


## 3. Définition du segment

Un segment représente la même combinaison métier suivie d'une année à l'autre.


In [46]:
# Colonnes qui identifient un segment.
SEGMENT = [
    "coo_id",
    "coa_id",
    "procedure_type",
    "app_type",
    "dec_level",
    "app_pc",
]

# Colonnes indispensables pour construire l'historique et la cible.
REQUIRED_COLUMNS = SEGMENT + ["year", "applied"]

# Vérifier que toutes les colonnes nécessaires existent.
missing_columns = [
    col for col in REQUIRED_COLUMNS
    if col not in df.columns
]

assert len(missing_columns) == 0, (
    f"Colonnes obligatoires absentes : {missing_columns}"
)

print("Contrôle des colonnes : OK")


Contrôle des colonnes : OK


In [47]:
# ============================================================
# 4. CONTRÔLES SUR YEAR ET APPLIED
# ============================================================

# Convertir year et applied en nombres.
# Une valeur non convertible devient NaN afin d'être détectée.
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["applied"] = pd.to_numeric(df["applied"], errors="coerce")

# Contrôles de validité.
assert df["year"].notna().all(), "Des années sont invalides."
assert df["applied"].notna().all(), "Des valeurs applied sont invalides."
assert (df["applied"] >= 0).all(), "Des volumes applied sont négatifs."

print("Période :", int(df["year"].min()), "-", int(df["year"].max()))
print("Doublons stricts :", int(df.duplicated().sum()))


Période : 2000 - 2025
Doublons stricts : 0


In [49]:
# ============================================================
# 5. STANDARDISATION DES CATÉGORIES
# ============================================================

# Standardiser les variables textuelles :
# espaces supprimés, majuscules et valeur UNKNOWN si manquante.
text_columns = [
    "procedure_type", "app_type", "dec_level", "app_pc",
    "origin_region", "asylum_region"
]

for col in text_columns:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.upper()
            .fillna("UNKNOWN")
        )

# Les IDs pays sont des catégories, pas des mesures numériques.
for col in ["coo_id", "coa_id"]:
    df[col] = df[col].astype("string").str.strip().fillna("UNKNOWN")


In [50]:
# ============================================================
# 6. UNICITÉ SEGMENT × ANNÉE
# ============================================================

# Une seule ligne doit représenter un segment pendant une année.
conflict_mask = df.duplicated(SEGMENT + ["year"], keep=False)
n_conflicts = int(conflict_mask.sum())

print("Lignes en conflit segment × année :", n_conflicts)

# Montrer les conflits au lieu de les supprimer silencieusement.
if n_conflicts > 0:
    display(
        df.loc[conflict_mask, SEGMENT + ["year", "applied"]]
        .sort_values(SEGMENT + ["year"])
        .head(20)
    )

assert n_conflicts == 0, (
    "Corriger les doublons segment × année avant de continuer."
)


Lignes en conflit segment × année : 0


## 7. Cible `applied_t1`

Pour une ligne de l'année T, `applied_t1` contient le volume du **même segment à T+1**.

Exemple : pour 2023, la cible est le volume observé en 2024.


In [51]:
# Créer une table temporaire avec la valeur future de applied.
target_table = df[SEGMENT + ["year", "applied"]].copy()

# Une valeur observée en 2024 doit être rattachée à la ligne 2023.
target_table["year"] = target_table["year"] - 1

# Renommer applied pour indiquer qu'il s'agit de la cible future.
target_table = target_table.rename(
    columns={"applied": "applied_t1"}
)

# LEFT JOIN : conserver toutes les observations de départ.
df = df.merge(
    target_table,
    on=SEGMENT + ["year"],
    how="left",
    validate="one_to_one"
)

print("Lignes avec cible T+1 :", int(df["applied_t1"].notna().sum()))


Lignes avec cible T+1 : 80032


## 8. `applied_lag1` : volume de l'année précédente

Pour une ligne 2023, `applied_lag1` doit contenir la valeur de 2022.

La jointure par année garantit qu'une année manquante ne sera pas confondue avec « la ligne précédente ».


In [52]:
# Copier l'historique de applied.
lag1_table = df[SEGMENT + ["year", "applied"]].copy()

# La valeur de 2022 doit être rattachée à la ligne 2023.
lag1_table["year"] = lag1_table["year"] + 1

# Renommer la valeur historique.
lag1_table = lag1_table.rename(
    columns={"applied": "applied_lag1"}
)

# Rattacher T-1 au même segment.
df = df.merge(
    lag1_table,
    on=SEGMENT + ["year"],
    how="left",
    validate="one_to_one"
)


## 9. `applied_lag2` : volume deux années auparavant


In [53]:
# Même logique que lag1, mais avec un décalage de deux années.
lag2_table = df[SEGMENT + ["year", "applied"]].copy()

# La valeur de 2021 doit être rattachée à la ligne 2023.
lag2_table["year"] = lag2_table["year"] + 2

lag2_table = lag2_table.rename(
    columns={"applied": "applied_lag2"}
)

df = df.merge(
    lag2_table,
    on=SEGMENT + ["year"],
    how="left",
    validate="one_to_one"
)

# Contrôle visuel de la chronologie.
display(
    df[
        SEGMENT
        + ["year", "applied_lag2", "applied_lag1", "applied", "applied_t1"]
    ].head(10)
)


,coo_id,coa_id,procedure_type,app_type,dec_level,app_pc,year,applied_lag2,applied_lag1,applied,applied_t1
0,2,11,G,A,AR,C,2006,NaN,NaN,14,6.0000
1,3,11,G,A,AR,C,2006,NaN,NaN,21,7.0000
2,4,11,G,A,AR,C,2006,NaN,NaN,5,5.0000
3,8,11,G,A,AR,C,2006,NaN,NaN,38,15.0000
4,14,11,G,A,AR,C,2006,NaN,NaN,11,5.0000
5,20,11,G,A,AR,C,2006,NaN,NaN,211,162.0000
6,24,11,G,A,AR,C,2006,NaN,NaN,5,NaN
7,27,11,G,A,AR,C,2006,NaN,NaN,5,NaN
8,30,11,G,A,AR,C,2006,NaN,NaN,5,NaN
9,32,11,G,A,AR,C,2006,NaN,NaN,8,NaN


## 10. Historique simple du segment


In [59]:
# Croissance entre T-1 et T.
# Si T-1 vaut zéro ou est absent, la croissance reste manquante.
df["growth_past_1"] = np.where(
    df["applied_lag1"] > 0,
    (df["applied"] - df["applied_lag1"]) / df["applied_lag1"],
    np.nan
)

# Croissance entre T-2 et T-1.
df["growth_past_2"] = np.where(
    df["applied_lag2"] > 0,
    (df["applied_lag1"] - df["applied_lag2"]) / df["applied_lag2"],
    np.nan
)

# Moyenne simple sur T, T-1 et T-2.
# Elle n'est calculée que si les trois années sont disponibles.
df["applied_mean_3y"] = (
    df[["applied", "applied_lag1", "applied_lag2"]]
    .sum(axis=1, min_count=3) / 3
)


## 11. Poids du corridor

Ces variables indiquent si le segment représente une petite ou une grande part des flux observés à T.

Elles supposent que l'année T est déjà consolidée au moment du scoring.


In [60]:
# Total à T pour le même pays d'origine.
df["origin_applied_total_t"] = (
    df.groupby(["coo_id", "year"])["applied"]
    .transform("sum")
)

# Total à T pour le même pays d'asile.
df["asylum_applied_total_t"] = (
    df.groupby(["coa_id", "year"])["applied"]
    .transform("sum")
)

# Part du segment dans le total de son pays d'origine.
df["segment_share_origin_t"] = np.where(
    df["origin_applied_total_t"] > 0,
    df["applied"] / df["origin_applied_total_t"],
    np.nan
)

# Part du segment dans le total de son pays d'asile.
df["segment_share_asylum_t"] = np.where(
    df["asylum_applied_total_t"] > 0,
    df["applied"] / df["asylum_applied_total_t"],
    np.nan
)


## 12. IDMC passé : contexte du pays d'origine à T−1


In [61]:
# Si le lag IDMC existe déjà, on le conserve.
if "idmc_total_origin_lag1" in df.columns:

    df["idmc_total_origin_lag1"] = pd.to_numeric(
        df["idmc_total_origin_lag1"],
        errors="coerce"
    )

# Sinon, le construire à partir de idmc_total_origin.
elif "idmc_total_origin" in df.columns:

    # Garder une table origine × année × IDMC.
    idmc_table = df[
        ["coo_id", "year", "idmc_total_origin"]
    ].drop_duplicates().copy()

    # Vérifier qu'il n'existe pas plusieurs valeurs IDMC
    # pour la même origine et la même année.
    conflicts = (
        idmc_table
        .groupby(["coo_id", "year"])["idmc_total_origin"]
        .nunique(dropna=True)
    )

    assert (conflicts <= 1).all(), (
        "Conflit IDMC pour une origine et une année."
    )

    # Une valeur IDMC de 2022 devient le lag1 de 2023.
    idmc_table["year"] = idmc_table["year"] + 1

    idmc_table = idmc_table.rename(
        columns={"idmc_total_origin": "idmc_total_origin_lag1"}
    )

    df = df.merge(
        idmc_table,
        on=["coo_id", "year"],
        how="left",
        validate="many_to_one"
    )

else:
    # Ne jamais inventer une valeur IDMC.
    df["idmc_total_origin_lag1"] = np.nan
    print("Attention : aucune donnée IDMC disponible.")


## 13. Décisions passées


In [62]:
# Features de décisions que nous souhaitons utiliser à T-1.
decision_features = [
    "decisions_total_lag1",
    "protection_rate_lag1",
    "rejection_rate_lag1",
]

# Si une colonne manque, elle reste NaN.
# NaN ne doit pas être remplacé arbitrairement par zéro.
for col in decision_features:
    if col not in df.columns:
        df[col] = np.nan
        print(f"Attention : {col} absente.")


Attention : decisions_total_lag1 absente.
Attention : protection_rate_lag1 absente.
Attention : rejection_rate_lag1 absente.


## 14. Liste finale des features


In [63]:
# Historique du segment.
history_features = [
    "applied",
    "applied_lag1",
    "applied_lag2",
    "growth_past_1",
    "growth_past_2",
    "applied_mean_3y",
]

# Poids du corridor.
corridor_features = [
    "origin_applied_total_t",
    "asylum_applied_total_t",
    "segment_share_origin_t",
    "segment_share_asylum_t",
]

# Contexte pays et métier.
context_features = [
    col for col in [
        "coo_id", "coa_id",
        "origin_region", "asylum_region",
        "procedure_type", "app_type",
        "dec_level", "app_pc"
    ]
    if col in df.columns
]

# Contexte IDMC.
idmc_features = ["idmc_total_origin_lag1"]

# Toutes les variables explicatives.
FEATURES = (
    ["year"]
    + context_features
    + history_features
    + corridor_features
    + idmc_features
    + decision_features
)

# Retirer d'éventuels noms répétés.
FEATURES = list(dict.fromkeys(FEATURES))

TARGET = "applied_t1"

print("Nombre de features :", len(FEATURES))
for i, feature in enumerate(FEATURES, 1):
    print(f"{i:02d}. {feature}")


Nombre de features : 23
01. year
02. coo_id
03. coa_id
04. origin_region
05. asylum_region
06. procedure_type
07. app_type
08. dec_level
09. app_pc
10. applied
11. applied_lag1
12. applied_lag2
13. growth_past_1
14. growth_past_2
15. applied_mean_3y
16. origin_applied_total_t
17. asylum_applied_total_t
18. segment_share_origin_t
19. segment_share_asylum_t
20. idmc_total_origin_lag1
21. decisions_total_lag1
22. protection_rate_lag1
23. rejection_rate_lag1


In [64]:
# ============================================================
# 15. CONTRÔLE ANTI-LEAKAGE
# ============================================================

# Ces variables contiennent une information future.
# Elles sont interdites dans les features.
FORBIDDEN = {"applied_t1", "growth_t1"}

leakage = FORBIDDEN.intersection(FEATURES)

assert len(leakage) == 0, (
    f"Target leakage détecté : {leakage}"
)

print("Contrôle anti-leakage : OK")


Contrôle anti-leakage : OK


In [65]:
# ============================================================
# 16. INDICATEUR MÉTIER FUTUR
# ============================================================

# growth_t1 sert uniquement à analyser les résultats.
# Il n'est PAS utilisé comme feature du modèle.
df["growth_t1"] = np.where(
    (df["applied"] > 0) & df["applied_t1"].notna(),
    (df["applied_t1"] - df["applied"]) / df["applied"],
    np.nan
)


## 17. Rapport Data Quality


In [71]:
# Mesurer la complétude des features et de la cible.
variables_to_check = FEATURES + [TARGET]

quality_report = pd.DataFrame({
    "variable": variables_to_check,
    "dtype": [str(df[col].dtype) for col in variables_to_check],
    "missing_n": [int(df[col].isna().sum()) for col in variables_to_check],
    "missing_pct": [
        round(df[col].isna().mean() * 100, 2)
        for col in variables_to_check
    ],
    "n_unique": [
        int(df[col].nunique(dropna=True))
        for col in variables_to_check
    ],
})

# Afficher en premier les variables les moins complètes.
quality_report = quality_report.sort_values(
    "missing_pct",
    ascending=False
)

display(quality_report)

# Conserver le rapport pour audit.
quality_report.to_csv(QUALITY_PATH, index=False)


,variable,dtype,missing_n,missing_pct,n_unique
22,rejection_rate_lag1,float64,120597,100.0000,0
21,protection_rate_lag1,float64,120597,100.0000,0
20,decisions_total_lag1,float64,120597,100.0000,0
19,idmc_total_origin_lag1,float64,74114,61.4600,500
14,applied_mean_3y,float64,59767,49.5600,6611
13,growth_past_2,float64,59767,49.5600,21435
11,applied_lag2,float64,52202,43.2900,3814
12,growth_past_1,float64,40565,33.6400,24684
10,applied_lag1,float64,40565,33.6400,4135
23,applied_t1,float64,40565,33.6400,4224


## 18. Dataset final et export


In [72]:
# Colonnes finales :
# identifiants du segment + features + cible + indicateur d'évaluation.
final_columns = (
    SEGMENT
    + ["year"]
    + FEATURES
    + [TARGET, "growth_t1"]
)

# Éviter les doublons de noms.
final_columns = list(dict.fromkeys(final_columns))

dataset_ml = df[final_columns].copy()

print("Dimensions :", dataset_ml.shape)
print(
    "Lignes entraînables :",
    int(dataset_ml[TARGET].notna().sum())
)
print(
    "Lignes sans cible T+1 :",
    int(dataset_ml[TARGET].isna().sum())
)

display(dataset_ml.head())

# Export du dataset.
dataset_ml.to_csv(OUTPUT_PATH, index=False)

print("Dataset exporté :", OUTPUT_PATH)
print("Rapport qualité :", QUALITY_PATH)


Dimensions : (120597, 25)
Lignes entraînables : 80032
Lignes sans cible T+1 : 40565


,coo_id,coa_id,procedure_type,app_type,dec_level,app_pc,year,origin_region,asylum_region,applied,applied_lag1,applied_lag2,growth_past_1,growth_past_2,applied_mean_3y,origin_applied_total_t,asylum_applied_total_t,segment_share_origin_t,segment_share_asylum_t,idmc_total_origin_lag1,decisions_total_lag1,protection_rate_lag1,rejection_rate_lag1,applied_t1,growth_t1
0,2,11,G,A,AR,C,2006,SOUTHERN ASIA,AUSTRALIA-NEW ZEALAND,14,NaN,NaN,NaN,NaN,NaN,15439,6546,0.0009,0.0021,NaN,NaN,NaN,NaN,6.0000,-0.5714
1,3,11,G,A,AR,C,2006,SOUTHERN EUROPE,AUSTRALIA-NEW ZEALAND,21,NaN,NaN,NaN,NaN,NaN,2232,6546,0.0094,0.0032,NaN,NaN,NaN,NaN,7.0000,-0.6667
2,4,11,G,A,AR,C,2006,NORTHERN AFRICA,AUSTRALIA-NEW ZEALAND,5,NaN,NaN,NaN,NaN,NaN,4272,6546,0.0012,0.0008,NaN,NaN,NaN,NaN,5.0000,0.0000
3,8,11,G,A,AR,C,2006,NORTHERN AFRICA,AUSTRALIA-NEW ZEALAND,38,NaN,NaN,NaN,NaN,NaN,2150,6546,0.0177,0.0058,NaN,NaN,NaN,NaN,15.0000,-0.6053
4,14,11,G,A,AR,C,2006,WESTERN ASIA,AUSTRALIA-NEW ZEALAND,11,NaN,NaN,NaN,NaN,NaN,35,6546,0.3143,0.0017,NaN,NaN,NaN,NaN,5.0000,-0.5455


Dataset exporté : ..\data\feature_engineering_outputs\dataset_ml_features.csv
Rapport qualité : ..\data\feature_engineering_outputs\data_quality_report.csv


# Étape suivante : modélisation

Le futur notebook de modélisation devra :

- effectuer un **split temporel** avant toute imputation ou tout encodage appris ;
- comparer les modèles à une baseline naïve `prediction_t1 = applied_t` ;
- évaluer les erreurs dans le temps et par pays/corridor ;
- vérifier que le ML apporte réellement une amélioration opérationnelle.

